In [ ]:
pip install -U langsmith openai

In [3]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
import anthropic

In [5]:
load_dotenv()

anthropic_api_key = os.getenv('API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')

In [7]:
from langsmith import Client, traceable

client = Client()

# Define dataset: these are your test cases
dataset = client.create_dataset(
    "Sample Dataset",
    description="A sample dataset in LangSmith.",
)

client.create_examples(
    inputs=[
        {"postfix": "to LangSmith"},
        {"postfix": "to Evaluations in LangSmith"},
    ],
    outputs=[
        {"response": "Welcome to LangSmith"},
        {"response": "Welcome to Evaluations in LangSmith"},
    ],
    dataset_id=dataset.id,
)

# Define an interface to your application (tracing optional)
@traceable
def dummy_app(inputs: dict) -> dict:
    return {"response": "Welcome " + inputs["postfix"]}

# Define your evaluator(s)
def exact_match(outputs: dict, reference_outputs: dict) -> bool:
    return outputs["response"] == reference_outputs["response"]

# Run the evaluation
experiment_results = client.evaluate(
    dummy_app, # Your AI system goes here
    data=dataset, # The data to predict and grade over
    evaluators=[exact_match], # The evaluators to score the results
    experiment_prefix="sample-experiment", # The name of the experiment
    metadata={"version": "1.0.0", "revision_id": "beta"}, # Metadata about the experiment
    max_concurrency=4,  # Add concurrency.
)

# Analyze the results via the UI or programmatically
# If you have 'pandas' installed you can view the results as a
# pandas DataFrame by uncommenting below:

experiment_results.to_pandas()

View the evaluation results for experiment: 'sample-experiment-0665cfd6' at:
https://smith.langchain.com/o/0667e4f5-8b9d-4268-90db-722304b71dc3/datasets/9ba4abec-2358-402c-93b9-7d35d711db60/compare?selectedSessions=bbb58c33-8c67-4502-9474-ea96e704290c




0it [00:00, ?it/s]

,inputs.postfix,outputs.response,error,reference.response,feedback.exact_match,execution_time,example_id,id
0,to Evaluations in LangSmith,Welcome to Evaluations in LangSmith,None,Welcome to Evaluations in LangSmith,True,0.019569,95c76272-1994-43c4-9ca2-e4732aab435b,1ba78af5-d84d-430c-b27b-18d20454f1b7
1,to LangSmith,Welcome to LangSmith,None,Welcome to LangSmith,True,0.019602,da87851e-5dea-45df-a73d-4f1198c2da0e,19bc46de-24a2-4a9b-8007-6e9f31603c03
